In [1]:
import requests
import zipfile
import io
import pandas as pd

HAGR_URLS = {
    "genage_human":    "https://genomics.senescence.info/genes/human_genes.zip",
    "genage_models":   "https://genomics.senescence.info/genes/model_organisms_genes.zip",
    "drugage":         "https://genomics.senescence.info/drugs/drugage.zip",
    "cellage":         "https://genomics.senescence.info/cells/cellage.zip",
    "longevitymap":    "https://genomics.senescence.info/longevity/longevity_genes.zip",
    "anage":           "https://genomics.senescence.info/species/dataset.zip",
    "gendr":           "https://genomics.senescence.info/diet/dataset.zip",
}

def normalize(df):
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

def inspect_db(name):
    print("\n" + "="*80)
    print(f"DATASET: {name}")
    print("="*80)

    url = HAGR_URLS[name]
    r = requests.get(url)

    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            print("\nFILES INSIDE ZIP:")
            for f in z.namelist():
                print(" -", f)

            csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]

            with z.open(csv_file) as f:
                df = pd.read_csv(f, on_bad_lines="skip")

    except:
        df = pd.read_csv(io.StringIO(r.text), on_bad_lines="skip")

    df = normalize(df)

    print("\nCOLUMNS:")
    for col in df.columns:
        print(" -", col)

    print("\nSAMPLE ROW:")
    print(df.iloc[0].to_dict())

    print("\nTOTAL ROWS:", len(df))


# RUN ALL
for db in HAGR_URLS:
    inspect_db(db)


DATASET: genage_human

FILES INSIDE ZIP:
 - genage_human.csv
 - release.html

COLUMNS:
 - genage_id
 - symbol
 - name
 - entrez_gene_id
 - uniprot
 - why

SAMPLE ROW:
{'genage_id': 1, 'symbol': 'GHR', 'name': 'growth hormone receptor', 'entrez_gene_id': 2690, 'uniprot': 'GHR_HUMAN', 'why': 'mammal'}

TOTAL ROWS: 307

DATASET: genage_models

COLUMNS:
 - <!doctype_html_public_"-//w3c//dtd_html_4.01_transitional//en"_"http://www.w3.org/tr/html4/loose.dtd">

SAMPLE ROW:
{'<!doctype_html_public_"-//w3c//dtd_html_4.01_transitional//en"_"http://www.w3.org/tr/html4/loose.dtd">': '<html>'}

TOTAL ROWS: 13

DATASET: drugage

COLUMNS:
 - <!doctype_html_public_"-//w3c//dtd_html_4.01_transitional//en"_"http://www.w3.org/tr/html4/loose.dtd">

SAMPLE ROW:
{'<!doctype_html_public_"-//w3c//dtd_html_4.01_transitional//en"_"http://www.w3.org/tr/html4/loose.dtd">': '<html>'}

TOTAL ROWS: 13

DATASET: cellage

COLUMNS:
 - <!doctype_html_public_"-//w3c//dtd_html_4.01_transitional//en"_"http://www.w3.org/tr

ParserError: Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.


In [3]:
import requests
import zipfile
import io
import pandas as pd

HAGR_URLS = {
    "genage_human":    "https://genomics.senescence.info/genes/human_genes.zip",
    "genage_models":   "https://genomics.senescence.info/genes/model_organisms_genes.zip",
    "drugage":         "https://genomics.senescence.info/drugs/drugage.zip",
    "cellage":         "https://genomics.senescence.info/cells/cellage.zip",
    "longevitymap":    "https://genomics.senescence.info/longevity/longevity_genes.zip",
    "anage":           "https://genomics.senescence.info/species/dataset.zip",
    "gendr":           "https://genomics.senescence.info/diet/dataset.zip",
}

def normalize(df):
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df

def inspect_db(name):
    print("\n" + "="*80)
    print(f"DATASET: {name}")
    print("="*80)

    url = HAGR_URLS[name]
    r = requests.get(url)

    if r.status_code != 200:
        print(" Download failed")
        return

    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            files = z.namelist()

            print("\n FILES INSIDE ZIP:")
            for f in files:
                print(" -", f)

            for f in files:
                print("\n" + "-"*60)
                print(f" READING FILE: {f}")
                print("-"*60)

                with z.open(f) as file:
                    content = file.read()

                    # Try decode safely
                    try:
                        text = content.decode("utf-8", errors="ignore")
                    except:
                        print(" Cannot decode file")
                        continue

                    # ---------- HTML ----------
                    if "<html" in text.lower():
                        print(" Detected HTML file")
                        try:
                            tables = pd.read_html(text)
                            print(f" Found {len(tables)} tables")

                            for i, df in enumerate(tables):
                                df = normalize(df)
                                print(f"\n--- TABLE {i} ---")
                                print("COLUMNS:", df.columns.tolist())
                                print("SAMPLE:", df.iloc[0].to_dict())
                                print("ROWS:", len(df))
                        except Exception as e:
                            print(" Failed to parse HTML:", e)

                    # ---------- TXT (tab-separated likely) ----------
                    elif f.endswith(".txt"):
                        print(" Detected TXT file")
                        try:
                            df = pd.read_csv(io.StringIO(text), sep="\t", on_bad_lines="skip")
                            df = normalize(df)

                            print("COLUMNS:", df.columns.tolist())
                            print("SAMPLE:", df.iloc[0].to_dict())
                            print("ROWS:", len(df))
                        except Exception as e:
                            print(" TXT parse failed:", e)

                    # ---------- CSV ----------
                    elif f.endswith(".csv"):
                        print(" Detected CSV file")
                        try:
                            df = pd.read_csv(io.StringIO(text), on_bad_lines="skip")
                            df = normalize(df)

                            print("COLUMNS:", df.columns.tolist())
                            print("SAMPLE:", df.iloc[0].to_dict())
                            print("ROWS:", len(df))
                        except Exception as e:
                            print(" CSV parse failed:", e)

                    else:
                        print(" Unknown file type")

    except zipfile.BadZipFile:
        print(" Not a ZIP — trying direct parse")

        text = r.text

        if "<html" in text.lower():
            print(" HTML detected")
            try:
                tables = pd.read_html(text)
                df = normalize(tables[0])

                print("COLUMNS:", df.columns.tolist())
                print("SAMPLE:", df.iloc[0].to_dict())
                print("ROWS:", len(df))
            except Exception as e:
                print(" HTML parse failed:", e)
        else:
            try:
                df = pd.read_csv(io.StringIO(text), on_bad_lines="skip")
                df = normalize(df)

                print("COLUMNS:", df.columns.tolist())
                print("SAMPLE:", df.iloc[0].to_dict())
                print("ROWS:", len(df))
            except Exception as e:
                print(" Raw parse failed:", e)


# RUN ALL
for db in HAGR_URLS:
    inspect_db(db)


DATASET: genage_human

 FILES INSIDE ZIP:
 - genage_human.csv
 - release.html

------------------------------------------------------------
 READING FILE: genage_human.csv
------------------------------------------------------------
 Detected CSV file
COLUMNS: ['genage_id', 'symbol', 'name', 'entrez_gene_id', 'uniprot', 'why']
SAMPLE: {'genage_id': 1, 'symbol': 'GHR', 'name': 'growth hormone receptor', 'entrez_gene_id': 2690, 'uniprot': 'GHR_HUMAN', 'why': 'mammal'}
ROWS: 307

------------------------------------------------------------
 READING FILE: release.html
------------------------------------------------------------
 Detected HTML file
 Failed to parse HTML: `Import lxml` failed.  Use pip or conda to install the lxml package.

DATASET: genage_models
 Download failed

DATASET: drugage
 Download failed

DATASET: cellage
 Download failed

DATASET: longevitymap

 FILES INSIDE ZIP:
 - longevity.csv
 - release.html

------------------------------------------------------------
 READI

In [1]:
import requests
import zipfile
import io
import pandas as pd

HAGR_URLS = {
    "genage_models":   "https://genomics.senescence.info/genes/model_organisms_genes.zip",
    "drugage":         "https://genomics.senescence.info/drugs/drugage.zip",
    "cellage":         "https://genomics.senescence.info/cells/cellage.zip",
    "anage":           "https://genomics.senescence.info/species/dataset.zip",
}

def inspect_dataset(name, url):
    print("\n" + "="*80)
    print(f"DATASET: {name}")
    print("="*80)

    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    content = r.content

    # -----------------------------------
    # CASE 1: NOT ZIP (HTML likely)
    # -----------------------------------
    if not zipfile.is_zipfile(io.BytesIO(content)):
        print("NOT A ZIP → likely HTML")

        text = content.decode("utf-8", errors="ignore")

        if "<html" in text.lower():
            print("HTML DETECTED")

            try:
                tables = pd.read_html(text)
                print(f"Tables found: {len(tables)}")

                for i, df in enumerate(tables[:2]):
                    print(f"\n--- TABLE {i} ---")
                    print("COLUMNS:", df.columns.tolist())
                    print("SAMPLE ROW:", df.iloc[0].to_dict())
                    print("ROWS:", len(df))

            except Exception as e:
                print("HTML parsing failed:", e)

        return

    # -----------------------------------
    # CASE 2: ZIP FILE
    # -----------------------------------
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        files = z.namelist()
        print("\nFILES INSIDE ZIP:")
        for f in files:
            print(" -", f)

        for f in files:
            print("\n" + "-"*60)
            print(f"READING FILE: {f}")
            print("-"*60)

            with z.open(f) as file:

                # TXT (ANAGE)
                if f.endswith(".txt"):
                    print("Detected TXT file")
                    df = pd.read_csv(file, sep="\t", on_bad_lines="skip")
                    print("COLUMNS:", df.columns.tolist())
                    print("SAMPLE:", df.iloc[0].to_dict())
                    print("ROWS:", len(df))

                # CSV
                elif f.endswith(".csv"):
                    print("Detected CSV file")
                    df = pd.read_csv(file, on_bad_lines="skip")
                    print("COLUMNS:", df.columns.tolist())
                    print("SAMPLE:", df.iloc[0].to_dict())
                    print("ROWS:", len(df))

                # HTML inside ZIP
                elif f.endswith(".html"):
                    print("Detected HTML file")
                    try:
                        text = file.read().decode("utf-8", errors="ignore")
                        tables = pd.read_html(text)
                        print(f"Tables found: {len(tables)}")
                        if tables:
                            df = tables[0]
                            print("COLUMNS:", df.columns.tolist())
                            print("SAMPLE:", df.iloc[0].to_dict())
                            print("ROWS:", len(df))
                    except Exception as e:
                        print("HTML parse failed:", e)


# RUN INSPECTION
for name, url in HAGR_URLS.items():
    inspect_dataset(name, url)


DATASET: genage_models
NOT A ZIP → likely HTML
HTML DETECTED
HTML parsing failed: [Errno 2] No such file or directory: <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">

<html>
<head>
	<title>Sorry, page not found</title>
</head>

<body bgColor=#ffffff>

<p><font size="5" color="#900030">Human Ageing Genomic Resources</font></p>

<div align="center"><p><hr color="#900030"></p>
		
<h2>Sorry, page not found (error 404)</h2>

<hr width="80%" size="1" noshade></div>

<p>The requested URL (http://genomics.senescence.info/genes/model_organisms_genes.zip) was not found. Your error has been logged and the server administrator informed.</p>

<p>Please check your spelling. If you continue seeing this error, it is possible the page you are looking for has been renamed, moved, or deleted. We apologize for any inconvenience. Please search the required information at the <a href="http://genomics.senescence.info">Human Ageing Genomic Resources</a>

In [2]:
import requests
import zipfile
import io
import pandas as pd

HAGR_URLS = {
    "genage_models": "https://genomics.senescence.info/genes/model_genes.zip",
    "drugage":       "https://genomics.senescence.info/drugs/dataset.zip",
    "cellage":       "https://genomics.senescence.info/cells/cellAge.zip",
    "anage":         "https://genomics.senescence.info/species/dataset.zip",
}

def inspect_dataset(name, url):
    print("\n" + "="*80)
    print(f"DATASET: {name}")
    print("="*80)

    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    content = r.content

    if not zipfile.is_zipfile(io.BytesIO(content)):
        print("❌ NOT A ZIP → still wrong URL or blocked")
        print(content[:500])
        return

    with zipfile.ZipFile(io.BytesIO(content)) as z:
        files = z.namelist()

        print("\nFILES INSIDE ZIP:")
        for f in files:
            print(" -", f)

        for f in files:
            print("\n" + "-"*60)
            print(f"READING FILE: {f}")
            print("-"*60)

            with z.open(f) as file:

                # TXT
                if f.endswith(".txt"):
                    print("Detected TXT")
                    df = pd.read_csv(file, sep="\t", on_bad_lines="skip")

                # CSV
                elif f.endswith(".csv"):
                    print("Detected CSV")
                    df = pd.read_csv(file, on_bad_lines="skip")

                else:
                    print("Skipping non-data file")
                    continue

                # Normalize preview only
                df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

                print("COLUMNS:", df.columns.tolist())
                print("SAMPLE:", df.iloc[0].to_dict())
                print("ROWS:", len(df))

In [3]:
for name, url in HAGR_URLS.items():
    inspect_dataset(name, url)


DATASET: genage_models
❌ NOT A ZIP → still wrong URL or blocked
b'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">\r\n\r\n<html>\r\n<head>\r\n\t<title>Sorry, page not found</title>\r\n</head>\r\n\r\n<body bgColor=#ffffff>\r\n\r\n<p><font size="5" color="#900030">Human Ageing Genomic Resources</font></p>\r\n\r\n<div align="center"><p><hr color="#900030"></p>\r\n\t\t\r\n<h2>Sorry, page not found (error 404)</h2>\r\n\r\n<hr width="80%" size="1" noshade></div>\r\n\r\n<p>The requested URL (http://genomics.senescence.info/genes/model_genes.zip) was '

DATASET: drugage

FILES INSIDE ZIP:
 - drugage.csv
 - release.html

------------------------------------------------------------
READING FILE: drugage.csv
------------------------------------------------------------
Detected CSV
COLUMNS: ['compound_name', 'species', 'strain', 'dosage', 'age_at_initiation', 'treatment_duration', 'avg_lifespan_change_percent', 'avg_lifespan_significance', 'ma